# 01_corpus_analysis

Simple corpus analysis for the law-domain project.

This notebook:
- loads JSONL files from the COLESLAW dataset,
- inspects sample records,
- computes basic corpus statistics,
- generates a few plots,
- runs a simple frequency analysis.

Expected project structure:
```text
01_corpus_analysis.ipynb
results/
  figures/
report/
  report.tex
```


## 1. Setup

Set `DATA_DIR` to the folder containing the downloaded COLESLAW JSONL files.

Examples:
- `../data/COLESLAW`
- `/path/to/COLESLAW`


In [ ]:
from pathlib import Path
import json
import re
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt

# Change this path to where your COLESLAW JSONL files are stored
DATA_DIR = Path('./data/COLESLAW')

# Output folders
RESULTS_DIR = Path('results')
FIGURES_DIR = RESULTS_DIR / 'figures'
RESULTS_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

MAX_DOCS = None  # Set to an integer like 50000 for faster testing

print('DATA_DIR =', DATA_DIR.resolve())
print('FIGURES_DIR =', FIGURES_DIR.resolve())


## 2. Find JSONL files

In [ ]:
jsonl_files = sorted(list(DATA_DIR.rglob('*.jsonl')))
print(f'Found {len(jsonl_files)} JSONL files')
for p in jsonl_files[:20]:
    print('-', p)

if not jsonl_files:
    raise FileNotFoundError(
        f'No JSONL files found under {DATA_DIR}. Update DATA_DIR to the dataset location.'
    )


## 3. Load records

The loader is intentionally flexible because field names can vary across sources.
It tries to extract text and source information from common keys.


In [ ]:
TEXT_KEYS = ['text', 'clean_text', 'content', 'body', 'document_text', 'full_text']
TITLE_KEYS = ['title', 'name', 'document_title']
SOURCE_KEYS = ['source', 'collection', 'subcorpus', 'domain']
ID_KEYS = ['id', 'document_id', 'doc_id', 'uuid']
DATE_KEYS = ['date', 'year', 'document_date', 'publication_date']

def first_present(record, keys, default=None):
    for key in keys:
        if key in record and record[key] not in [None, '']:
            return record[key]
    return default

def normalize_text(x):
    if x is None:
        return ''
    if isinstance(x, list):
        x = ' '.join(str(item) for item in x)
    return str(x).strip()

records = []
loaded = 0

for file_path in jsonl_files:
    fallback_source = file_path.stem
    with open(file_path, 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                continue

            text = normalize_text(first_present(rec, TEXT_KEYS, default=''))
            title = normalize_text(first_present(rec, TITLE_KEYS, default=''))
            source = normalize_text(first_present(rec, SOURCE_KEYS, default=fallback_source))
            doc_id = normalize_text(first_present(rec, ID_KEYS, default=f'{file_path.name}:{line_num}'))
            date = normalize_text(first_present(rec, DATE_KEYS, default=''))

            records.append({
                'doc_id': doc_id,
                'source': source,
                'title': title,
                'date': date,
                'text': text,
                'n_chars': len(text),
                'n_words': len(text.split()),
                'file_name': file_path.name,
                'raw_keys': sorted(rec.keys()),
            })
            loaded += 1
            if MAX_DOCS is not None and loaded >= MAX_DOCS:
                break
    if MAX_DOCS is not None and loaded >= MAX_DOCS:
        break

df = pd.DataFrame(records)
print(f'Loaded {len(df):,} records')
df.head()


## 4. Inspect sample records

In [ ]:
display_cols = ['doc_id', 'source', 'title', 'date', 'n_words', 'file_name']
df[display_cols].sample(min(5, len(df)), random_state=42)


In [ ]:
sample_idx = 0
print('Source:', df.loc[sample_idx, 'source'])
print('Title:', df.loc[sample_idx, 'title'])
print('Date:', df.loc[sample_idx, 'date'])
print('Words:', df.loc[sample_idx, 'n_words'])
print('Keys:', df.loc[sample_idx, 'raw_keys'])
print('\nTEXT SAMPLE:\n')
print(df.loc[sample_idx, 'text'][:2000])


## 5. Basic statistics

In [ ]:
summary = {
    'n_documents': int(len(df)),
    'n_sources': int(df['source'].nunique()),
    'total_words': int(df['n_words'].sum()),
    'avg_words_per_doc': float(df['n_words'].mean()),
    'median_words_per_doc': float(df['n_words'].median()),
    'min_words_per_doc': int(df['n_words'].min()),
    'max_words_per_doc': int(df['n_words'].max()),
}
summary_df = pd.DataFrame([summary])
summary_df


In [ ]:
source_counts = (
    df.groupby('source')
      .agg(n_documents=('doc_id', 'count'),
           total_words=('n_words', 'sum'),
           avg_words=('n_words', 'mean'))
      .sort_values('n_documents', ascending=False)
      .reset_index()
)
source_counts


In [ ]:
source_counts.to_csv(RESULTS_DIR / 'source_counts.csv', index=False)
summary_df.to_csv(RESULTS_DIR / 'summary_stats.csv', index=False)
print('Saved summary tables to results/')


## 6. Plots

In [ ]:
# Documents by source
ax = source_counts.plot(x='source', y='n_documents', kind='bar', legend=False, figsize=(10, 5))
ax.set_title('Number of documents by source')
ax.set_xlabel('Source')
ax.set_ylabel('Documents')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'documents_by_source.png', dpi=200, bbox_inches='tight')
plt.show()


In [ ]:
# Document length distribution
ax = df['n_words'].plot(kind='hist', bins=50, figsize=(10, 5))
ax.set_title('Document length distribution')
ax.set_xlabel('Number of words')
ax.set_ylabel('Frequency')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'document_length_histogram.png', dpi=200, bbox_inches='tight')
plt.show()


In [ ]:
# Log-scale histogram for long-tail distributions
ax = df['n_words'].plot(kind='hist', bins=50, figsize=(10, 5), log=True)
ax.set_title('Document length distribution (log frequency)')
ax.set_xlabel('Number of words')
ax.set_ylabel('Log frequency')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'document_length_histogram_log.png', dpi=200, bbox_inches='tight')
plt.show()


## 7. Frequency analysis

This is a simple baseline frequency analysis. You can later improve it with a Slovenian stopword list if needed.


In [ ]:
BASIC_STOPWORDS = {
    'in', 'je', 'da', 'se', 'na', 'za', 's', 'z', 'so', 'ki', 'v', 'o', 'po', 'od', 'do', 'ter',
    'kot', 'ali', 'pa', 'ob', 'pri', 'iz', 'a', 'to', 'ta', 'te', 'ti', 'the', 'and', 'of', 'for',
    'is', 'are', 'with', 'on', 'by', 'an', 'be', 'or', 'as', 'that', 'this', 'it'
}

def tokenize(text):
    text = text.lower()
    tokens = re.findall(r'\b\w+\b', text, flags=re.UNICODE)
    return [t for t in tokens if len(t) > 2 and t not in BASIC_STOPWORDS and not t.isdigit()]

token_counter = Counter()
for text in df['text']:
    token_counter.update(tokenize(text))

top_terms = pd.DataFrame(token_counter.most_common(30), columns=['term', 'count'])
top_terms


In [ ]:
top_terms.to_csv(RESULTS_DIR / 'top_terms.csv', index=False)

ax = top_terms.head(20).plot(x='term', y='count', kind='bar', legend=False, figsize=(12, 5))
ax.set_title('Top 20 frequent terms')
ax.set_xlabel('Term')
ax.set_ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'top_terms.png', dpi=200, bbox_inches='tight')
plt.show()


## 8. Quick observations for the report

After running the notebook, fill in a few short observations here and transfer them into `report/report.tex`.


In [ ]:
print('Suggested things to note in the report:')
print('- number of loaded documents')
print('- which sources dominate the corpus')
print('- whether document lengths vary a lot')
print('- whether metadata seems consistent across files')
print('- whether the most frequent terms match your expected legal domain vocabulary')
